# Combined coolin operation optimization

In [ ]:
# - [ ] Model water availability in reservoir as a function of precipitations


In [1]:
import os
MR = "/home/patomareao/MATLAB/R2024b"
os.environ['LD_LIBRARY_PATH'] = f"{MR}/runtime/glnxa64:\
{MR}/bin/glnxa64:\
{MR}/sys/os/glnxa64:\
{MR}/extern/bin/glnxa64:\
/lib/x86_64-linux-gnu:"


In [2]:
from pathlib import Path
import itertools
from collections import deque
from dataclasses import asdict
import json
import hjson
import numpy as np
import pandas as pd
from loguru import logger
import pygmo as pg
from IPython.display import clear_output

from solhycool_modeling import OperationPoint, EnvironmentVariables
from solhycool_optimization.utils.evaluation import optimize, evaluate_global_algos
from solhycool_optimization.utils import CustomEncoder
from solhycool_optimization.visualization import plot_algo_comparison
from solhycool_visualization.diagrams import WascopStateVisualizer
from solhycool_visualization.operation import plot_results

# Visualization packages
from phd_visualizations.optimization import plot_obj_scape_comp_1d
from phd_visualizations import save_figure

import combined_cooler_model
cc_model = combined_cooler_model.initialize()

logger.disable("phd_visualizations")

data_path: Path = Path("../../data")
env_path: Path = data_path / "datasets/environment_data_20220101_20241231.h5"
base_output_path: Path = Path("./algo_comparison")

%load_ext autoreload
%autoreload 2

# Constants
Vavail0 = 1 # m³, Set a value so that at midday the more expensive source will be used


In [3]:
from solhycool_optimization.problems.horizon import CombinedCoolerProblem as Problem
system_id = "cc_horizon"


In [4]:
# Load environemental data

# Experimental data
df_env_exp = pd.read_csv(Path("../../modeling/assets/data.csv"), )

# Load environment into EnvironmentVariables for the episode
df_env = pd.read_hdf(env_path)

selected_date_str: str = "20220103" 
df_day = df_env.loc[selected_date_str]

output_path = base_output_path / selected_date_str
if not output_path.exists():
    output_path.mkdir(parents=True)


### Single evaluation

In [7]:
operation_pt_list, algo_list, pop_list, fitness_list, fitness_history_list = optimize(
    problem, algo_id="ipopt", 
    n_trials = 1, log_verbosity = 1, extra_outputs=True, 
    max_iter=50, use_mbh=True, use_cstrs=False,
    initial_pop_size=100, wrapper_algo_iters=10, 
)


2025-03-24 09:38:42.202 | INFO     | solhycool_optimization.utils.evaluation:optimize:153 - Iteration 1 out of 1


J=11.81, 2.07 s
J=15.32, 0.12 s
J=9.63, 0.13 s
J=11.05, 0.16 s
J=8.94, 0.15 s
J=11.62, 0.15 s
J=9.58, 0.13 s
J=12.14, 0.14 s
J=8.40, 0.12 s
J=10.83, 0.12 s
J=11.16, 0.14 s
J=11.08, 0.12 s
J=9.06, 0.24 s
J=10.59, 0.13 s
J=13.75, 0.13 s
J=11.20, 0.15 s
J=12.53, 0.14 s
J=12.41, 0.17 s
J=12.06, 0.14 s
J=6.81, 0.12 s
J=13.60, 0.16 s
J=10.53, 0.15 s
J=7.77, 0.14 s
J=8.65, 0.17 s
J=10.59, 0.16 s
J=9.84, 0.11 s
J=12.71, 0.13 s
J=14.10, 0.13 s
J=8.53, 0.11 s
J=12.46, 0.12 s
J=11.00, 0.13 s
J=11.08, 0.11 s
J=11.18, 0.13 s
J=11.04, 0.12 s
J=11.08, 0.15 s
J=11.86, 0.12 s
J=9.72, 0.13 s
J=11.46, 0.13 s
J=10.29, 0.13 s
J=11.10, 0.17 s
J=8.36, 0.10 s
J=11.64, 0.12 s
J=11.96, 0.12 s
J=7.98, 0.10 s
J=11.68, 0.13 s
J=11.05, 0.13 s
J=9.11, 0.13 s
J=9.28, 0.13 s
J=10.60, 0.11 s
J=7.17, 0.11 s
J=10.43, 0.13 s
J=10.12, 0.12 s
J=9.02, 0.16 s
J=9.79, 0.12 s
J=11.60, 0.12 s
J=9.54, 0.13 s
J=10.87, 0.13 s
J=7.13, 0.11 s
J=9.71, 0.14 s
J=9.86, 0.12 s
J=10.58, 0.10 s
J=10.85, 0.10 s
J=12.92, 0.11 s
J=9.03, 0.10 s

: 

: 

In [5]:
env_vars = EnvironmentVariables.from_dataframe(df_day)
env_vars.Vavail = [float(Vavail0)] * len(env_vars.Tamb)
env_vars.Pw_s2 = env_vars.Pe * 2 * 1e-2 # Alternative source incurs in a cost similar to electricity

# display(env_vars)

problem = Problem(env_vars=env_vars)



In [10]:
results = evaluate_global_algos(
    problem=problem,
    n_trials=1,
    max_n_obj_fun_evals=10_000,
    algo_ids=["ihs"],
    use_cstr=[True],
    pop_size=[400],
    log_verbosity=[50]
)
results


2025-03-24 10:39:32.830 | INFO     | solhycool_optimization.utils.evaluation:evaluate_global_algos:217 - Running ihs with cstr=True, pop_size=400 and max_iter=1000
2025-03-24 10:39:32.830 | INFO     | solhycool_optimization.utils.evaluation:optimize:153 - Iteration 1 out of 1


The command was cancelled. 


TypeError: cannot unpack non-iterable bool object

### Full day using simulation data

# Old

In [ ]:
from itertools import product
# Try with validation data
df = pd.read_csv(Path("../../modeling/assets/data.csv"), )
# df = df[(df["Rp"]>0.9)] #& (df["Rs"]<0.1)]

# ds = df.iloc[0]
display(df)
detailed_list = []

for idx, (price_factor, df_idx) in enumerate(product([1000, 100, 1], range(len(df)))):
    ds = df.iloc[df_idx]
    env_vars = EnvironmentVariables(
        HR=ds["HR"],
        Tamb=ds["Tamb"],
        Q=ds["Qreleased"],
        # mv=ds["Pth_kW"] / (w_props(T=ds["Tv_C"]+273.15, x=1).h - w_props(T=ds["Tv_C"]+273.15, x=0).h) * 3600,
        Tv=ds["Tv"],
        Pw_s1=df_env["water_price_morocco_eur_l"]*price_factor,
        Pw_s2=df_env["water_price_morocco_alternative_eur_l"]*price_factor,
        Pe=cost_e,
        Vavail=df_env["Vavail_l"]
    )

    problem = Problem(env_vars=env_vars, debug_mode=False)

    if idx == 0:
        # Evalute prior to optimization (using experimental values), only once
        dv = problem.decision_vector_to_decision_variables([ds["qc"], ds["wwct"]])
        # display(dv)
        ev = env_vars.to_matlab()
        _, _, detailed = cc_model.combined_cooler_model(ev.Tamb, ev.HR, ev.mv, dv.qc, dv.Rp, dv.Rs, dv.wdc, dv.wwct, ev.Tv, nargout=3)
        detailed_list.append(detailed)

    # Evaluate optimization
    detailed_list.append(optimize(problem, extra_outputs=False, log_verbosity=0, n_trials=1)[0])

df_results = pd.DataFrame(detailed_list)
display(df_results)

fig = plot_hydraulic_distribution(df_results["qc"].values, df_results["Rp"].values, df_results["Rs"].values)
display(fig)
